
# Day 2 · 실습 ⑦. ResNet과 ViT의 이미지 처리 비교

**연계 이론:** 이론 2. ResNet과 현대적인 CNN 아키텍처 / 이론 3. Vision Transformer의 구조 / 이론 4. Self-Attention의 계산 원리

## 학습 목표

이번 실습에서는 사전학습된 ResNet18과 ViT-B/16을 사용하여 동일한 이미지를 처리하고, 두 모델의 특징 추출 방식과 출력 구조를 비교합니다.

1. ResNet18과 ViT-B/16의 사전학습 가중치를 불러오고 모델별 전처리를 적용합니다.
2. 두 모델의 이미지 분류 결과를 확인하고 비교합니다.
3. ResNet의 Feature Map과 ViT의 Patch Token 표현을 확인합니다.
4. CNN과 Vision Transformer의 이미지 처리 방식 차이를 설명합니다.

## 실습 진행 순서

1. PyTorch 환경 및 GPU 확인
2. 사전학습 모델 준비
3. 입력 이미지와 전처리
4. 이미지 분류 결과 비교
5. ResNet Feature Map 추출 및 시각화
6. ViT Token 출력 확인 및 시각화
7. 전체 결과 비교 및 정리

> 이번 실습에서는 모델을 새로 학습하지 않습니다. 사전학습 모델의 추론 결과와 중간 특징 표현을 관찰합니다.

In [ ]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from urllib.request import urlretrieve
from PIL import Image
from torch.nn import functional as F

from torchvision.models import (
    resnet18,
    vit_b_16,
    ResNet18_Weights,
    ViT_B_16_Weights,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("실행 장치:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU 메모리:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3, 2
        ),
        "GB"
    )


## 1. 사전학습 ResNet18과 ViT-B/16 준비

이번 실습에서는 TorchVision이 제공하는 두 가지 이미지 분류 모델을 사용합니다.

| 모델 | 주요 특징 | 입력 크기 |
|---|---|---|
| ResNet18 | Residual Block을 이용하는 CNN | 224×224 |
| ViT-B/16 | 16×16 Patch와 Transformer Encoder를 사용하는 ViT | 224×224 |

두 모델 모두 ImageNet-1K 데이터셋으로 학습된 가중치를 사용합니다.

ImageNet-1K 분류 모델의 최종 출력은 1000개 클래스에 대한 점수입니다.

`eval()`은 모델을 추론 모드로 전환하는 메서드입니다.

사전학습 가중치를 처음 사용하는 경우에는 파일이 다운로드되며, 이후에는 PyTorch 캐시에 저장된 가중치를 재사용할 수 있습니다.

In [ ]:
# 사전학습 가중치 선택
resnet_weights = ResNet18_Weights.IMAGENET1K_V1
vit_weights = ViT_B_16_Weights.IMAGENET1K_V1

# 사전학습 모델 생성
# 최초 실행 시 가중치를 다운로드할 수 있습니다.
resnet_model = resnet18(
    weights=resnet_weights
)

vit_model = vit_b_16(
    weights=vit_weights
)

# 실행 장치 이동 및 추론 모드
resnet_model = resnet_model.to(device).eval()
vit_model = vit_model.to(device).eval()

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

print("ResNet18 파라미터:",
      f"{count_parameters(resnet_model):,}")

print("ViT-B/16 파라미터:",
      f"{count_parameters(vit_model):,}")

print("\n모델 준비 완료")


## 2. 입력 이미지 준비

두 모델에 동일한 이미지를 입력하여 결과를 비교합니다.

기본 예제는 PyTorch에서 제공하는 강아지 사진입니다.

개인 이미지를 사용하려면 아래 코드의 `USER_IMAGE_PATH`를 변경할 수 있습니다.

예를 들어 Notebook과 같은 폴더에 `my_photo.jpg`가 있다면 다음과 같이 설정합니다.

`USER_IMAGE_PATH = "my_photo.jpg"`

이미지를 변경할 때에는 이후의 전처리·추론·시각화 셀을 다시 실행합니다.

In [ ]:
# 변경해 보기
USER_IMAGE_PATH = ""

if USER_IMAGE_PATH.strip():
    image_path = Path(USER_IMAGE_PATH)

    if not image_path.is_file():
        raise FileNotFoundError(
            f"이미지를 찾을 수 없습니다: {image_path}"
        )
else:
    image_path = Path("day2_07_sample_dog.jpg")

    if not image_path.exists():
        sample_url = (
            "https://raw.githubusercontent.com/"
            "pytorch/hub/master/images/dog.jpg"
        )

        try:
            urlretrieve(sample_url, image_path)
        except Exception as exc:
            raise RuntimeError(
                "예제 이미지를 다운로드하지 못했습니다. "
                "이미지 파일을 Notebook 폴더에 저장하고 "
                "USER_IMAGE_PATH를 지정하세요."
            ) from exc

image = Image.open(image_path).convert("RGB")

print("이미지 경로:", image_path)
print("원본 크기:", image.size)
print("이미지 모드:", image.mode)

plt.figure(figsize=(6, 5))
plt.imshow(image)
plt.title("Original Image")
plt.axis("off")
plt.show()


## 3. 모델별 이미지 전처리

사전학습 모델은 학습에 사용된 입력 조건에 맞추어 이미지를 전처리해야 합니다.

TorchVision에서는 사전학습 가중치 객체의 `transforms()`를 통해 권장 전처리 방법을 가져올 수 있습니다.

전처리에는 이미지 크기 조절, 중앙 자르기, Tensor 변환, 정규화 등이 포함됩니다.

이번에 사용하는 두 가중치의 기본 전처리는 모두 224×224 입력을 생성합니다.

전처리 후 Tensor Shape은 다음과 같습니다.

`(N, C, H, W) = (1, 3, 224, 224)`

- N: 배치 크기
- C: RGB 채널 수
- H: 이미지 높이
- W: 이미지 너비

In [ ]:
# 모델별 권장 전처리
resnet_preprocess = resnet_weights.transforms()
vit_preprocess = vit_weights.transforms()

# 같은 원본 이미지에 각각 적용
resnet_input = (
    resnet_preprocess(image)
    .unsqueeze(0)
    .to(device)
)

vit_input = (
    vit_preprocess(image)
    .unsqueeze(0)
    .to(device)
)

print("ResNet18 입력 Shape:",
      tuple(resnet_input.shape))

print("ViT-B/16 입력 Shape:",
      tuple(vit_input.shape))

assert resnet_input.shape == (1, 3, 224, 224)
assert vit_input.shape == (1, 3, 224, 224)

print("\nResNet18 전처리:")
print(resnet_preprocess)

print("\nViT-B/16 전처리:")
print(vit_preprocess)


## 4. 모델의 중간 특징 추출

두 모델의 차이를 살펴보기 위해 최종 예측뿐 아니라 중간 계층의 출력도 확인합니다.

### ResNet18

ResNet18에서는 네 개의 Stage 출력을 추출합니다.

- `layer1`
- `layer2`
- `layer3`
- `layer4`

각 출력은 `(N, C, H, W)` 형태의 Feature Map입니다.

### ViT-B/16

ViT에서는 다음 출력을 추출합니다.

- `conv_proj`: 이미지 Patch Embedding을 계산하는 단계
- `encoder`: Transformer Encoder의 최종 Token 표현

ViT의 Encoder 출력은 `(N, Token 수, Embedding 차원)` 형태입니다.

중간 출력을 확인하기 위해 PyTorch의 `register_forward_hook()`을 사용합니다.

Forward Hook은 모델의 순전파 과정에서 특정 계층의 출력을 가져오는 기능입니다.

In [ ]:
# 중간 특징을 저장할 딕셔너리
features = {}

def save_feature(name):
    def hook(module, inputs, output):
        features[name] = output.detach().cpu()
    return hook


handles = []

# ResNet18의 Stage 출력
for layer_name in [
    "layer1", "layer2", "layer3", "layer4"
]:
    layer = getattr(resnet_model, layer_name)

    handles.append(
        layer.register_forward_hook(
            save_feature(f"resnet_{layer_name}")
        )
    )

# ViT Patch Embedding 출력
handles.append(
    vit_model.conv_proj.register_forward_hook(
        save_feature("vit_patch_embedding")
    )
)

# ViT Encoder 출력
handles.append(
    vit_model.encoder.register_forward_hook(
        save_feature("vit_encoder")
    )
)

# 추론 실행
try:
    with torch.inference_mode():
        resnet_logits = resnet_model(resnet_input)
        vit_logits = vit_model(vit_input)
finally:
    # Forward Hook 제거
    for handle in handles:
        handle.remove()

print("ResNet18 최종 출력:",
      tuple(resnet_logits.shape))

print("ViT-B/16 최종 출력:",
      tuple(vit_logits.shape))

assert resnet_logits.shape == (1, 1000)
assert vit_logits.shape == (1, 1000)

print("\n중간 특징 추출 완료")


## 5. 이미지 분류 결과 비교

두 모델의 최종 출력은 클래스별 Logit입니다.

Softmax를 적용하면 각 클래스에 대한 예측 확률 형태로 변환할 수 있습니다.

이번 실습에서는 두 모델의 Top-5 예측을 비교합니다.

확인할 내용은 다음과 같습니다.

1. 두 모델의 1순위 예측 클래스가 같은가요?
2. Top-5에 포함된 클래스는 어떻게 다른가요?
3. 같은 이미지를 처리했는데도 예측 확률이 다르게 나타나는 이유는 무엇일까요?

두 모델은 서로 다른 구조와 학습된 가중치를 사용하므로, 클래스별 출력값이 달라질 수 있습니다.

In [ ]:
def get_top5(logits, weights):
    probabilities = torch.softmax(
        logits, dim=1
    )[0].detach().cpu()

    top_probs, top_indices = torch.topk(
        probabilities, k=5
    )

    categories = weights.meta["categories"]

    return [
        (categories[idx.item()], prob.item())
        for idx, prob in zip(top_indices, top_probs)
    ]


resnet_top5 = get_top5(
    resnet_logits, resnet_weights
)

vit_top5 = get_top5(
    vit_logits, vit_weights
)

print("=== ResNet18 Top-5 ===")
for rank, (label, prob) in enumerate(resnet_top5, 1):
    print(f"{rank}. {label}: {prob:.2%}")

print("\n=== ViT-B/16 Top-5 ===")
for rank, (label, prob) in enumerate(vit_top5, 1):
    print(f"{rank}. {label}: {prob:.2%}")


# 예측 결과 시각화
fig, axes = plt.subplots(
    1, 2, figsize=(14, 5)
)

for ax, results, title in zip(
    axes,
    [resnet_top5, vit_top5],
    ["ResNet18", "ViT-B/16"]
):
    labels = [item[0] for item in results][::-1]
    probabilities = [
        item[1] for item in results
    ][::-1]

    ax.barh(labels, probabilities)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Probability")
    ax.set_title(f"{title} Top-5")

plt.tight_layout()
plt.show()


## 6. ResNet18의 Feature Map 확인

ResNet18은 Convolution을 이용하여 이미지 특징을 추출합니다.

앞서 저장한 `layer1`부터 `layer4`까지의 출력 Shape을 확인합니다.

일반적인 224×224 입력에서 각 Stage의 출력은 다음과 같습니다.

| Stage | 출력 Shape |
|---|---|
| Layer 1 | `(1, 64, 56, 56)` |
| Layer 2 | `(1, 128, 28, 28)` |
| Layer 3 | `(1, 256, 14, 14)` |
| Layer 4 | `(1, 512, 7, 7)` |

Stage를 거치면서 공간 크기는 줄어들고 특징 채널 수는 증가합니다.

시각화에서는 각 Stage의 여러 채널을 평균하여 하나의 2차원 맵으로 표현합니다.

이 결과는 모델이 계산한 특징을 관찰하기 위한 것으로, 객체의 위치를 정확히 설명하는 근거 지도는 아닙니다.

In [ ]:
layer_names = [
    "layer1", "layer2", "layer3", "layer4"
]

fig, axes = plt.subplots(
    1, 4, figsize=(16, 4)
)

for ax, layer_name in zip(axes, layer_names):
    feature_map = features[f"resnet_{layer_name}"]

    print(
        layer_name,
        "Shape:",
        tuple(feature_map.shape)
    )

    # 배치 첫 번째 이미지
    # 채널별 Feature Map을 평균하여 표현
    mean_map = feature_map[0].mean(dim=0).numpy()

    ax.imshow(mean_map, cmap="viridis")
    ax.set_title(
        f"{layer_name}\n"
        f"{feature_map.shape[2]} x "
        f"{feature_map.shape[3]}"
    )
    ax.axis("off")

plt.suptitle("ResNet18 Feature Maps")
plt.tight_layout()
plt.show()


## 7. ViT의 Patch Embedding과 Token 출력 확인

ViT-B/16은 이미지를 16×16 Patch로 나눕니다.

224×224 입력은 다음과 같이 처리됩니다.

`224 ÷ 16 = 14`

따라서 이미지 Patch 개수는 다음과 같습니다.

`14 × 14 = 196`

각 Patch는 768차원 Embedding으로 변환됩니다.

CLS Token이 추가되면 전체 Token 수는 197개가 됩니다.

TorchVision ViT-B/16의 중간 출력은 다음과 같습니다.

| 단계 | 출력 Shape |
|---|---|
| Patch Embedding(Conv 출력) | `(1, 768, 14, 14)` |
| Transformer Encoder | `(1, 197, 768)` |
| CLS Token | `(1, 768)` |
| 이미지 Patch Token | `(1, 196, 768)` |

CNN의 Feature Map은 공간적 배열을 가진 Tensor이고, ViT의 Encoder 출력은 Token 시퀀스라는 차이가 있습니다.

In [ ]:
# Patch Embedding 출력
vit_patch_feature = features["vit_patch_embedding"]

# Transformer Encoder 출력
vit_tokens = features["vit_encoder"]

print(
    "Patch Embedding Shape:",
    tuple(vit_patch_feature.shape)
)

print(
    "Encoder 출력 Shape:",
    tuple(vit_tokens.shape)
)

# 첫 번째 Token은 CLS Token
cls_token = vit_tokens[:, 0, :]

# 나머지는 이미지 Patch Token
patch_tokens = vit_tokens[:, 1:, :]

print(
    "CLS Token Shape:",
    tuple(cls_token.shape)
)

print(
    "Patch Token Shape:",
    tuple(patch_tokens.shape)
)

assert vit_patch_feature.shape == (1, 768, 14, 14)
assert vit_tokens.shape == (1, 197, 768)
assert cls_token.shape == (1, 768)
assert patch_tokens.shape == (1, 196, 768)


## 8. ViT의 Patch 특징 시각화

ViT의 Token 표현을 관찰하는 방법에는 여러 가지가 있습니다.

이번 실습에서는 두 가지 방법을 사용합니다.

**첫째, Patch Embedding의 특징 크기를 확인합니다.**

Transformer Encoder에 입력되기 전 Patch Embedding 출력에서 채널 방향의 특징 크기를 계산합니다.

**둘째, CLS Token과 이미지 Patch Token의 유사도를 계산합니다.**

Transformer Encoder를 통과한 CLS Token과 각 Patch Token 사이의 Cosine Similarity를 계산합니다.

196개의 Patch별 유사도 값을 14×14 배열로 변환하여 시각화합니다.

이 결과는 Token 표현 사이의 유사도를 나타내며, 앞선 실습 ⑥에서 계산한 Attention Weight와는 다른 값입니다.

또한 예측에 대한 중요도 지도나 실제 Attention Heatmap을 의미하지 않습니다.

In [ ]:
# 1. Patch Embedding 특징 크기
patch_embedding_map = (
    vit_patch_feature[0]
    .square()
    .mean(dim=0)
    .sqrt()
    .numpy()
)

# 2. CLS Token과 Patch Token의 Cosine Similarity
cls = vit_tokens[0, 0, :]
patches = vit_tokens[0, 1:, :]

similarities = F.cosine_similarity(
    patches,
    cls.unsqueeze(0),
    dim=1
)

similarity_map = (
    similarities.reshape(14, 14).numpy()
)

# ViT에 입력된 이미지의 정규화를 해제하여 표시
vit_mean = torch.tensor(
    vit_preprocess.mean
).view(3, 1, 1)

vit_std = torch.tensor(
    vit_preprocess.std
).view(3, 1, 1)

vit_display_image = (
    vit_input[0].detach().cpu() * vit_std
    + vit_mean
).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(
    1, 3, figsize=(15, 5)
)

axes[0].imshow(vit_display_image)
axes[0].set_title("ViT Input Image")
axes[0].axis("off")

im1 = axes[1].imshow(
    patch_embedding_map,
    cmap="viridis"
)
axes[1].set_title("Patch Embedding Magnitude")
axes[1].axis("off")
fig.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(
    similarity_map,
    cmap="coolwarm"
)
axes[2].set_title("CLS-Patch Cosine Similarity")
axes[2].axis("off")
fig.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()


## 9. ResNet18과 ViT-B/16의 처리 결과 비교

지금까지 두 모델의 예측과 중간 특징을 확인했습니다.

| 비교 항목 | ResNet18 | ViT-B/16 |
|---|---|---|
| 주요 연산 | Convolution | Self-Attention |
| 이미지 특징 표현 | Feature Map | Patch Token |
| 중간 특징 구조 | `(N, C, H, W)` | `(N, Tokens, D)` |
| 특징 결합 방식 | 지역적 특징의 계층적 결합 | Token 간 관계를 통한 정보 결합 |
| 최종 출력 | ImageNet-1K 클래스 점수 | ImageNet-1K 클래스 점수 |

두 모델 모두 이미지 분류를 수행하지만, 내부 특징의 표현과 처리 과정은 다릅니다.

ResNet18에서는 Feature Map의 채널 수와 공간 크기가 변화하는 과정을 확인했습니다.

ViT-B/16에서는 196개의 이미지 Patch Token과 하나의 CLS Token이 Transformer Encoder를 통과하는 구조를 확인했습니다.

In [ ]:
print("=" * 55)
print("ResNet18 vs ViT-B/16 비교 결과")
print("=" * 55)

print("\n[최종 분류 결과]")
print(
    "ResNet18:",
    resnet_top5[0][0],
    f"({resnet_top5[0][1]:.2%})"
)

print(
    "ViT-B/16:",
    vit_top5[0][0],
    f"({vit_top5[0][1]:.2%})"
)

print("\n[중간 특징 표현]")
print(
    "ResNet Layer 4:",
    tuple(features["resnet_layer4"].shape)
)
print(
    "ViT Encoder:",
    tuple(features["vit_encoder"].shape)
)

print("\n[모델 파라미터]")
print(
    "ResNet18:",
    f"{count_parameters(resnet_model):,}"
)
print(
    "ViT-B/16:",
    f"{count_parameters(vit_model):,}"
)

print("\n실습 완료")


## 10. 실습 내용 정리와 핵심 질문

### 10.1 이번 실습에서 확인한 내용

**첫째, 두 모델 모두 동일한 이미지를 분류할 수 있습니다.**

ResNet18과 ViT-B/16은 모두 ImageNet-1K 데이터셋으로 사전학습된 모델이며, 1000개 클래스에 대한 예측 점수를 출력합니다.

**둘째, CNN과 ViT는 특징을 표현하는 방식이 다릅니다.**

ResNet18은 합성곱 연산을 통해 Feature Map을 생성합니다.

ViT-B/16은 이미지를 Patch로 분할하고 Token Embedding으로 변환한 다음 Transformer Encoder에서 처리합니다.

**셋째, 모델의 내부 출력은 서로 다른 구조를 갖습니다.**

CNN의 Feature Map은 채널·높이·너비를 가진 Tensor입니다.

ViT의 Token 표현은 Token 개수와 Embedding 차원을 가진 Tensor입니다.

### 10.2 핵심 질문

1. ResNet18이 이미지 특징을 추출할 때 사용하는 핵심 연산은 무엇인가요?
2. ResNet18에서 `layer1`부터 `layer4`까지 공간 크기는 어떻게 변하나요?
3. ViT-B/16이 224×224 이미지를 분할하면 몇 개의 이미지 Patch가 만들어지나요?
4. ViT-B/16의 Transformer Encoder 출력 Shape은 무엇인가요?
5. CNN의 Feature Map과 ViT의 Token Embedding은 어떤 차이가 있나요?

### 10.3 조건 변경 및 결과 관찰

이번에는 입력 이미지를 다른 사진으로 변경하고, 전처리부터 예측 및 중간 특징 추출까지 다시 실행합니다.

결과를 비교할 때에는 다음 사항을 확인합니다.

- 입력 이미지가 달라지면 두 모델의 분류 결과가 어떻게 변하는가?
- ResNet의 Feature Map에는 어떤 변화가 나타나는가?
- ViT의 Patch Token 표현과 CLS-Patch 유사도 분포는 어떻게 달라지는가?

두 모델은 학습한 특징과 구조가 다르므로 같은 입력 이미지에서도 서로 다른 결과를 만들 수 있습니다.

### 10.4 다음 학습 연결

이번 실습에서는 사전학습 ResNet18과 ViT-B/16을 이용하여 이미지 분류 결과와 내부 특징 표현을 비교했습니다.

다음에는 이러한 사전학습 모델이 기존에 학습한 특징을 새로운 이미지 분류 문제에 활용하는 방법을 배우고, Feature Extraction과 Fine-tuning의 차이를 살펴봅니다.